# 05 · Stable softmax, log-sum-exp and online softmax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/05_stable_softmax.ipynb) &nbsp; 📖 Concept deck: [Softmax and log-sum-exp](https://nilesh-salpe.github.io/visual-ai/math/08-softmax-logsumexp/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** why the naive softmax overflows, the max-subtraction fix, a stable log-sum-exp and log-softmax, and the one-pass online softmax that FlashAttention builds on.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

## Task 1: Softmax that never overflows

Naively, `np.exp(x) / np.exp(x).sum()` overflows in float32 once a logit exceeds about 88.7. Subtracting the maximum first changes nothing mathematically (the factor $e^{-m}$ cancels) but keeps every exponent ≤ 0.
Implement it for 1-D float32 input and return float32.

In [ ]:
def softmax_stable(x):
    x = np.asarray(x, np.float32)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_softmax_stable(f):
    x = np.array([1000., 1001., 1002.], np.float32)
    with np.errstate(over='ignore', invalid='ignore'):
        naive = np.exp(x) / np.exp(x).sum()
    assert np.isnan(naive).all(), "(sanity: the naive version breaks here)"
    y = f(x)
    assert np.all(np.isfinite(y)) and np.isclose(y.sum(), 1)
    assert np.allclose(y, f(x - 1000)), "softmax is shift-invariant"

check("softmax_stable", _test_softmax_stable, softmax_stable)

## Task 2: Log-sum-exp and log-softmax

$\mathrm{LSE}(x) = \log \sum_i e^{x_i} = m + \log \sum_i e^{x_i - m}$ with $m = \max x$. Then $\log \mathrm{softmax}(x)_i = x_i - \mathrm{LSE}(x)$.
Cross-entropy losses use log-softmax directly, never `log(softmax(x))`, which turns tiny probabilities into `log(0) = -inf`.

In [ ]:
def log_softmax(x):
    x = np.asarray(x, float)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_log_softmax(f):
    x = np.array([0., 800., 1.])
    y = f(x)
    assert np.all(np.isfinite(y)), "must stay finite even when a probability underflows"
    assert np.isclose(y[0], -800), "log p(0) = 0 - LSE ≈ -800"
    z = np.random.default_rng(0).normal(size=6)
    assert np.allclose(np.exp(f(z)), np.exp(z) / np.exp(z).sum())

check("log_softmax", _test_log_softmax, log_softmax)

## Task 3: Online softmax in one pass over blocks

Read the logits block by block, keeping only a running max `m` and running sum `d` of $e^{x - m}$. When a block raises the max to `m'`, rescale the old sum by $e^{m - m'}$ (Milakov & Gimelshein, 2018).
Return `(m, d)` after all blocks; the softmax is then $e^{x - m}/d$ and $\mathrm{LSE} = m + \log d$.

In [ ]:
def online_softmax(x, block=4):
    m, d = -np.inf, 0.0
    # TODO: for each block: new max, rescale d, add the block's terms
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_online_softmax(f):
    x = np.random.default_rng(1).normal(size=37) * 5
    for b in [1, 4, 10, 37]:
        m, d = f(x, b)
        assert np.isclose(m, x.max()), "m must end as the global max"
        assert np.isclose(m + np.log(d), np.log(np.sum(np.exp(x)))), f"block={b}: LSE mismatch"

check("online_softmax", _test_online_softmax, online_softmax)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Softmax that never overflows
def softmax_stable_solution(x):
    x = np.asarray(x, np.float32)
    e = np.exp(x - x.max())
    return e / e.sum()

check("softmax_stable (reference)", _test_softmax_stable, softmax_stable_solution)

In [ ]:
# @title Solution for Task 2: Log-sum-exp and log-softmax
def log_softmax_solution(x):
    x = np.asarray(x, float); m = x.max()
    return x - (m + np.log(np.sum(np.exp(x - m))))

check("log_softmax (reference)", _test_log_softmax, log_softmax_solution)

In [ ]:
# @title Solution for Task 3: Online softmax in one pass over blocks
def online_softmax_solution(x, block=4):
    m, d = -np.inf, 0.0
    for i in range(0, len(x), block):
        blk = x[i:i + block]; m2 = max(m, blk.max())
        d = d * np.exp(m - m2) + np.sum(np.exp(blk - m2)); m = m2
    return m, d

check("online_softmax (reference)", _test_online_softmax, online_softmax_solution)

---
## Experiments
**Where formats overflow.** `exp(x)` overflows FP16 (max 65,504) once x > ln 65504 ≈ 11.09, and FP32 once x > ≈ 88.72. Attention logits easily exceed 11, which is one reason softmax runs with max subtraction and in higher precision.

In [ ]:
for dt in [np.float16, np.float32]:
    lim = np.log(np.finfo(dt).max)
    with np.errstate(over='ignore'):
        print(f"{dt.__name__}: max {float(np.finfo(dt).max):.4g}, exp overflows above x ≈ {lim:.2f}; exp({lim + 0.1:.2f}) = {np.exp(dt(lim + 0.1))}")

---
## Takeaways
* Subtract the max before exponentiating: same softmax, no overflow.
* Use log-sum-exp / log-softmax for losses instead of log(softmax).
* The online softmax keeps a running max and rescaled sum, so softmax can be computed block by block, the basis of FlashAttention.